# Data Ingestion to VectorDB pipeline

## Ingestion

In [6]:
## read all files from pdf folder
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"found {len(pdf_files)} pdf document(s)")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # add more metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = "pdf"
            
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"Exception: {e}")
        
        print(f"Total documents loaded: {len(all_documents)}")
        return all_documents



C:\Users\abhis\AppData\Local\Temp\ipykernel_21672\3531924799.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\Users\abhis\source\repos\RAGChatBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
all_pdf_docs = process_all_pdfs("../data/pdf_files")

found 1 pdf document(s)

Processing: driving test manual.pdf
loaded 55 pages
Total documents loaded: 55


## Splitting/Chunking

In [8]:
# Text Splitting: To break the document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n", "\n\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    # example chunk
    if split_docs:
        print(f"example chunk: {split_docs[0].page_content[:200]}")

    return split_docs





In [9]:
chunks = split_documents(all_pdf_docs)


split 55 documents into 159 chunks
example chunk: i
For additional information about Ohio’s BMV services, 
visit the BMV website at www.bmv.ohio.gov.
ABOUT THIS MANUAL
The Ohio Bureau of Motor Vehicles (BMV) oversees driver and motor 
vehicle licensi


## Embedding and Vector Store


In [10]:
# we'll use an opensource embedding model: sentence-transformers from huggingface and an opensource vectordb: faiss-cpu or chromadb

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity # for similarity search on stored vectors 

### Class for Embedding

In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedidng manager
        
        args: model_name: HuggingFace model name used for sentence embeddings

        """

        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"loading the embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded successfully. Embedding dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates embeddings for a list of text values.

        Args: 
            texts: List of text values to embed

        Returns:
            A numpy array of embeddings with shape: (len(texts), embedding_dimension)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts/chunks")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings
    
## initialise Embeddingmanager class
embedding_manager = EmbeddingManager()



loading the embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4729.37it/s]


model loaded successfully. Embedding dimensions: 384


### Class for VectorStore

In [12]:
import os


class VectorStore:
    """Manages document embeddings in a chromadb vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initializes the vector db

        Args:
            collection_name: Name of the chromaDb collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialise_store()

    def initialise_store(self):
        """Initialise chromadb client and collection"""

        try:
            # create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create a collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF Doc embeddings for RAG"}
             )
            
            print(f"Vector Store initialised. Collection: {self.collection_name}")
            print(f"Existing Documents in collection: {self.collection.count()}")
        except Exception as e:
           print(f"Exception occured: {e}")
           raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the VectorStore

        Args: 
            documents: List of LangChain documents
            embeddings: corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to the vector store.")

        # prepare data for chromadb
        ids = []
        metadata_list = []
        documents_texts = []
        embeddings_list = []

        # assigning a uuid to each record
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # generate unique uuid
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadata_list.append(metadata)

            # Document Content
            documents_texts.append(doc.page_content)

            # Embedding data
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try: 
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadata_list,
                documents=documents_texts
            )
            print(f"Successfully added {len(documents_texts)} records to vector store")
            print(f"Total records in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error occured: {e}")
            raise

    def clear_db(self):
        self.client.delete_collection(self.collection_name)

vectorStore = VectorStore()


Vector Store initialised. Collection: pdf_documents
Existing Documents in collection: 159


### Actual embedding of chunks and storing data in vectorStore


In [13]:
# apply embedding to existing chunks that were created earlier

texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

# store in vector db
vectorStore.add_documents(chunks, embeddings)

Generating embeddings for 159 texts/chunks


Batches: 100%|██████████| 5/5 [00:04<00:00,  1.21it/s]


Generated embeddings with shape: (159, 384)
Adding 159 documents to the vector store.
Successfully added 159 records to vector store
Total records in collection: 318


In [14]:
## clearing the vector store 
# vectorStore.clear_db()

# Retrieval Pipeline (from vectorStore based on similarity score)

In [15]:
class RAGRetriever:
    """Handles query based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialise the retriever
        
        Args:
            vector_store: Vector Store containing documents embedding
            embedding_manager: Manager for generating embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        Args:
            query: the search query 
            top_k: num of top results to return
            score_threshold: minimum similarity score to consider a match
        
        Returns:
            List of Dictionaries containing retreived documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"top_k: {top_k}, score_threshold: {score_threshold}")

        # Embed the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in vector store based on the generated embedding 
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # process search results
            retrieved_docs = []

            # the result from querying the collection will have an internal keys 'documents', 'metadatas' and 'distances'
            if results['documents'] and results['documents'][0]:
                ids = results['ids'][0]
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} records after filtering.")
            else: 
                print("No matching records found")
            
            return retrieved_docs
        
        except Exception as e:
            print("Exception occured: {e}")
            return []

rag_retriever = RAGRetriever(vectorStore, embedding_manager)





In [18]:
rag_retriever.retrieve("What is the legal allowed drinking permission?")

Retrieving documents for query: 'What is the legal allowed drinking permission?'
top_k: 5, score_threshold: 0.0
Generating embeddings for 1 texts/chunks


Batches: 100%|██████████| 1/1 [00:00<00:00, 88.59it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 records after filtering.


[{'id': 'doc_2a3d1961_74',
  'content': '25\nAlcohol and the Law\x07\nIn Ohio, the legal drinking age is 21 years or older and it is against the law to operate a motor vehicle with \na blood-alcohol concentration (BAC) of:\n•\t.08% or higher at any age \n•\t.04% or higher for commercial drivers  \n•\t.02% or higher when under the age 21\nIf an individual is stopped for suspected driving under the influence of drugs and/or alcohol, or physical \ncontrol of a vehicle while under the influence, the officer will conduct sobriety tests and request a \nchemical test to determine the alcohol and/or drug content in the blood. Evidence of impaired driving is \nbased on physical findings by the arresting officer and the results of a blood, breath, plasma, or urine \ntest. \nAdministrative License Suspension (ALS) Test Over the Limit\x03 — If an individual consents to a chemical \ntest within two hours of the arrest and the test results show a BAC of .08 or higher, the arresting officer',
  'meta

# Integrating LLM to the retrieved context for final output

In [ ]:
# Integrate Claude LLM to give final outputs

from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
import os
load_dotenv()

## Initialize yout LLM here (set your API key in environment)

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    api_key="api-key",
    max_tokens=1024)

def retrieve_with_llm(query, retriever, llm, top_k=3):
    # retrieve the context
    results = retriever.retrieve(query, top_k)
    context = "\n\n".join(doc['content'] for doc in results) if results else ""

    if not context:
        return "No relevant context found to answer the question"
    
    # generate answer using LLM
    prompt = f"""
                Use the following context to answer the question concisely.
                Context: {context}
                Question: {query}
                Answer: 
                """
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content




In [ ]:
answer = retrieve_with_llm("What is the legal age to drive in Ohio?", rag_retriever, llm)